In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from xgboost import XGBClassifier

# Custom Transformer for Outlier Handling
class OutlierHandling(BaseEstimator, TransformerMixin):
    def __init__(self, method='percentile', lower_percentile=0.01, upper_percentile=0.99):
        self.method = method
        self.lower_percentile = lower_percentile
        self.upper_percentile = upper_percentile

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if self.method == 'percentile':
            return self.handle_outliers_percentile(X)
        elif self.method == 'iqr':
            return self.handle_outliers_iqr(X)
        else:
            return X

    # Percentile Outlier Handling
    def handle_outliers_percentile(self, df):
        lower_bound = df.quantile(self.lower_percentile)
        upper_bound = df.quantile(self.upper_percentile)
        return df.clip(lower=lower_bound, upper=upper_bound, axis=1)

    # IQR Outlier Handling
    def handle_outliers_iqr(self, df):
        Q1 = df.quantile(0.25)
        Q3 = df.quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        return df.clip(lower=lower_bound, upper=upper_bound, axis=1)

# Load sample dataset
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Set up a DataFrame to log results
results = pd.DataFrame(columns=['Transformation', 'Method', 'Outlier_Method', 'Imputation', 'Accuracy'])

# Function to log results
def log_results(transformation, method, outlier_method, imputation, accuracy):
    results.loc[len(results)] = [transformation, method, outlier_method, imputation, accuracy.mean()]

# Models and outlier handling methods to experiment with
models = [
    RandomForestClassifier(random_state=42),
    XGBClassifier(random_state=42),
    ExtraTreesClassifier(random_state=42)
]
outlier_methods = ['percentile', 'iqr']
imputers = [
    ('SimpleImputer_Median', SimpleImputer(strategy='median')),
    ('KNNImputer', KNNImputer(n_neighbors=5))
]

# Iterate over outlier handling methods and imputation strategies
for model in models:
    # Step 1: Log accuracy before any transformations (Raw data)
    pipeline_raw = Pipeline(steps=[
        ('scaler', StandardScaler()),  # Only scaling raw data
        ('model', model)
    ])

    # Evaluate with cross-validation on raw data
    accuracy_raw = cross_val_score(pipeline_raw, X, y, cv=5, scoring='accuracy')
    log_results('Before_Transformation', model.__class__.__name__, 'None', 'None', accuracy_raw)

    # Step 2: After Feature Transformation
    for outlier_method in outlier_methods:
        for imputer_name, imputer in imputers:
            # Set up pipeline with outlier handling, imputation, and scaling
            pipeline_transformed = Pipeline(steps=[
                ('outlier_handling', OutlierHandling(method=outlier_method)),  # Outlier handling
                ('imputer', imputer),  # Imputation
                ('scaler', StandardScaler()),  # Scaling
                ('model', model)  # Model training
            ])

            # Evaluate with cross-validation on transformed data
            accuracy_transformed = cross_val_score(pipeline_transformed, X, y, cv=5, scoring='accuracy')
            log_results('After_Transformation', model.__class__.__name__, outlier_method, imputer_name, accuracy_transformed)

    # Step 3: After Feature Transformation + Feature Construction (e.g., Polynomial Features)
    for outlier_method in outlier_methods:
        for imputer_name, imputer in imputers:
            # Set up pipeline with polynomial features
            pipeline_feature_construction = Pipeline(steps=[
                ('outlier_handling', OutlierHandling(method=outlier_method)),  # Outlier handling
                ('imputer', imputer),  # Imputation
                ('scaler', StandardScaler()),  # Scaling
                ('poly_features', PolynomialFeatures(degree=2, include_bias=False)),  # Polynomial features
                ('model', model)  # Model training
            ])

            # Evaluate with cross-validation after feature construction
            accuracy_construction = cross_val_score(pipeline_feature_construction, X, y, cv=5, scoring='accuracy')
            log_results('After_Transformation+Feature_Construction', model.__class__.__name__, outlier_method, imputer_name, accuracy_construction)

# Display results
print(results.sort_values(by='Accuracy', ascending=False))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from xgboost import XGBClassifier  # Make sure to install xgboost if not installed: pip install xgboost

# Custom Transformer for Outlier Handling
class OutlierHandling(BaseEstimator, TransformerMixin):
    def __init__(self, method='percentile', lower_percentile=0.01, upper_percentile=0.99):
        self.method = method
        self.lower_percentile = lower_percentile
        self.upper_percentile = upper_percentile

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if self.method == 'percentile':
            return self.handle_outliers_percentile(X)
        elif self.method == 'iqr':
            return self.handle_outliers_iqr(X)
        else:
            return X

    # Percentile Outlier Handling
    def handle_outliers_percentile(self, df):
        lower_bound = df.quantile(self.lower_percentile)
        upper_bound = df.quantile(self.upper_percentile)
        return df.clip(lower=lower_bound, upper=upper_bound, axis=1)

    # IQR Outlier Handling
    def handle_outliers_iqr(self, df):
        Q1 = df.quantile(0.25)
        Q3 = df.quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        return df.clip(lower=lower_bound, upper=upper_bound, axis=1)

# Load sample dataset
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Set up a DataFrame to log results
results = pd.DataFrame(columns=['Method', 'Outlier_Method', 'Imputation', 'Accuracy'])

# Function to log results
def log_results(method, outlier_method, imputation, accuracy):
    results.loc[len(results)] = [method, outlier_method, imputation, accuracy.mean()]

# Models and outlier handling methods to experiment with
models = [
    RandomForestClassifier(random_state=42),
    XGBClassifier(random_state=42),
    ExtraTreesClassifier(random_state=42)
]
outlier_methods = ['percentile', 'iqr']
imputers = [
    ('SimpleImputer_Median', SimpleImputer(strategy='median')),
    ('KNNImputer', KNNImputer(n_neighbors=5))
]

# Iterate over outlier handling methods and imputation strategies
for outlier_method in outlier_methods:
    for imputer_name, imputer in imputers:
        for model in models:
            # Set up pipeline
            pipeline = Pipeline(steps=[
                ('outlier_handling', OutlierHandling(method=outlier_method)),  # Outlier handling step
                ('imputer', imputer),  # Imputation step
                ('scaler', StandardScaler()),  # Feature scaling
                ('model', model)  # Model training
            ])

            # Evaluate with cross-validation
            accuracy = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')

            # Log results
            log_results(model.__class__.__name__, outlier_method, imputer_name, accuracy)

# Display results
print(results.sort_values(by='Accuracy', ascending=False))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

X = df[num_cols]
y = df['loan_status'].replace({'Y': 1, 'N': 0})

# Set up a DataFrame to log results
results = pd.DataFrame(columns=['Model', 'Missing Value Handling', 'Accuracy'])

# Function to log results
def log_results(model_name, imputation, accuracy):
    results.loc[len(results)] = [model_name, imputation, accuracy.mean()]

# Define the models to experiment with
models = {
    'Logistic Regression': LogisticRegression(),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Support Vector Machine': SVC(),
    'KNN': KNeighborsClassifier(),
    'Decision Trees': DecisionTreeClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42),
    'Extra Trees': ExtraTreesClassifier(random_state=42)
}

# Define the imputation strategies to experiment with
imputers = [
    ('SimpleImputer_Median', SimpleImputer(strategy='median')),
    ('KNNImputer', KNNImputer(n_neighbors=5))
]

# Iterate over imputation strategies and models
for imputer_name, imputer in imputers:
    for model_name, model in models.items():
        # Set up pipeline
        pipeline = Pipeline(steps=[
            ('imputer', imputer),  # Imputation step
            ('scaler', StandardScaler()),  # Scaling step (optional)
            ('model', model)  # Model training step
        ])

        # Evaluate with cross-validation
        accuracy = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')

        # Log results
        log_results(model_name, imputer_name, accuracy)

# Display results sorted by accuracy
print(results.sort_values(by='Accuracy', ascending=False))


# Library imports and Configurations


In [ ]:
# Upgrade scikit-learn
!pip install --upgrade scikit-learn -q

# Install flash library
!pip install git+https://github.com/flash-lib/flash.git -q

In [ ]:
# Standard Libraries
import os
import json
import math

# Data Manipulation and Preprocessing
import numpy as np
import pandas as pd

# Data Preprocessing and Transformation
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    FunctionTransformer, PowerTransformer, QuantileTransformer,
    StandardScaler, MinMaxScaler, RobustScaler,
    LabelEncoder, OneHotEncoder
)

# Machine Learning Models
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier,
    VotingClassifier
)

# Model Evaluation and Hyperparameter Tuning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Saving
import joblib

import flash as fz

In [ ]:
# Change directory to current working directory
%cd /content/drive/MyDrive/Projects/loan-sanction-prediction

# Accuracy test

## Handling Missing values

In [ ]:
# Load the dataset
clean_df = pd.read_csv('data/interim/cleaned_data_v1.csv')
unclean_df = pd.read_csv('data/loan_sanction')


## Feature Construction


# Building predictive model


## Preparing the data


In [ ]:
df_constructed = pd.read_csv('data/interim/feature_constructed.csv')
df_cleaned = pd.read_csv('data/interim/cleaned_data_v1.csv')

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

def compare_model_accuracy(X_original, X_constructed, y, models=None, test_size=0.2, random_state=42):
    """
    Compares the accuracy of models using original features and constructed features.

    Parameters:
    - X_original: pd.DataFrame, Features before feature construction
    - X_constructed: pd.DataFrame, Features after feature construction
    - y: pd.Series or np.array, Target variable
    - models: dict, Optional. Dictionary of models to test (e.g., {'LogisticRegression': LogisticRegression()})
    - test_size: float, Fraction of data to use for testing
    - random_state: int, Random state for reproducibility

    Returns:
    - results: pd.DataFrame, Accuracy comparison between original and constructed features
    """

    # Default models to test if not provided
    if models is None:
        models = {
            'LogisticRegression': LogisticRegression(),
            'DecisionTree': DecisionTreeClassifier(),
            'RandomForest': RandomForestClassifier()
        }

    # Split the data into training and testing sets
    X_train_orig, X_test_orig, y_train, y_test = train_test_split(X_original, y, test_size=test_size, random_state=random_state)
    X_train_const, X_test_const = train_test_split(X_constructed, test_size=test_size, random_state=random_state)[0:2]

    # Dictionary to store accuracy results
    accuracy_results = {
        'Model': [],
        'Accuracy with Original Features': [],
        'Accuracy with Constructed Features': []
    }

    # Iterate over each model and compare accuracies
    for model_name, model in models.items():
        # Accuracy with original features
        model.fit(X_train_orig, y_train)
        y_pred_orig = model.predict(X_test_orig)
        accuracy_orig = accuracy_score(y_test, y_pred_orig)

        # Accuracy with constructed features
        model.fit(X_train_const, y_train)
        y_pred_const = model.predict(X_test_const)
        accuracy_const = accuracy_score(y_test, y_pred_const)

        # Store the results
        accuracy_results['Model'].append(model_name)
        accuracy_results['Accuracy with Original Features'].append(accuracy_orig)
        accuracy_results['Accuracy with Constructed Features'].append(accuracy_const)

    # Convert the results into a DataFrame for easy viewing
    results_df = pd.DataFrame(accuracy_results)

    return results_df


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

def compare_model_accuracy(X_original, X_constructed, y, models=None, test_size=0.2, random_state=42):
    """
    Compares the accuracy of models using original features and constructed features with preprocessing.

    Parameters:
    - X_original: pd.DataFrame, Features before feature construction
    - X_constructed: pd.DataFrame, Features after feature construction
    - y: pd.Series or np.array, Target variable
    - categorical_features: list of str, Names of categorical feature columns
    - numerical_features: list of str, Names of numerical feature columns
    - models: dict, Optional. Dictionary of models to test (e.g., {'LogisticRegression': LogisticRegression()})
    - test_size: float, Fraction of data to use for testing
    - random_state: int, Random state for reproducibility

    Returns:
    - results: pd.DataFrame, Accuracy comparison between original and constructed features
    """

    # Default models to test if not provided
    if models is None:
        models = {
            'LogisticRegression': LogisticRegression(),
            'DecisionTree': DecisionTreeClassifier(),
            'RandomForest': RandomForestClassifier()
        }

    # Preprocessing pipelines for original and constructed features
    def create_preprocessor(df):
        num_cols, cat_cols, other_cols = fz.extract_features(df, 'all')
        return ColumnTransformer(
            transformers=[
                ('num', Pipeline(steps=[
                    ('imputer', SimpleImputer(strategy='median')),
                    ('scaler', StandardScaler())
                ]), num_cols),
                ('cat', Pipeline(steps=[
                    ('imputer', SimpleImputer(strategy='most_frequent')),
                    ('onehot', OneHotEncoder(handle_unknown='ignore'))
                ]), cat_cols)
            ]
        )

    # Split the data into training and testing sets
    X_train_orig, X_test_orig, y_train, y_test = train_test_split(X_original, y, test_size=test_size, random_state=random_state)
    X_train_const, X_test_const = train_test_split(X_constructed, test_size=test_size, random_state=random_state)[0:2]

    preprocessor_orig = create_preprocessor(X_train_orig)
    preprocessor_const = create_preprocessor(X_train_const)

    # Label encode the target feature
    le = LabelEncoder()
    y_train = le.fit_transform(y_train)
    y_test = le.transform(y_test)

    # Dictionary to store accuracy results
    accuracy_results = {
        'Model': [],
        'Accuracy with Original Features': [],
        'Accuracy with Constructed Features': []
    }

    # Iterate over each model and compare accuracies
    for model_name, model in models.items():
        # Accuracy with original features
        X_train_orig_trans = preprocessor_orig.fit_transform(X_train_orig)
        X_test_orig_trans = preprocessor_orig.transform(X_test_orig)
        model.fit(X_train_orig_trans, y_train)
        y_pred_orig = model.predict(X_test_orig_trans)
        accuracy_orig = accuracy_score(y_test, y_pred_orig)

        # Accuracy with constructed features
        X_train_const_trans = preprocessor_const.fit_transform(X_train_const)
        X_test_const_trans = preprocessor_const.transform(X_test_const)
        model.fit(X_train_const_trans, y_train)
        y_pred_const = model.predict(X_test_const_trans)
        accuracy_const = accuracy_score(y_test, y_pred_const)

        # Store the results
        accuracy_results['Model'].append(model_name)
        accuracy_results['Accuracy with Original Features'].append(accuracy_orig)
        accuracy_results['Accuracy with Constructed Features'].append(accuracy_const)

    # Convert the results into a DataFrame for easy viewing
    results_df = pd.DataFrame(accuracy_results)

    return results_df

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
import numpy as np

def compare_model_accuracy(X_original, X_constructed, y, models=None, test_size=0.2, random_state=42, cv=5):
    """
    Compares the accuracy of models using original features and constructed features with preprocessing,
    and performs cross-validation.

    Parameters:
    - X_original: pd.DataFrame, Features before feature construction
    - X_constructed: pd.DataFrame, Features after feature construction
    - y: pd.Series or np.array, Target variable
    - models: dict, Optional. Dictionary of models to test (e.g., {'LogisticRegression': LogisticRegression()})
    - test_size: float, Fraction of data to use for testing (not used in cross-validation)
    - random_state: int, Random state for reproducibility
    - cv: int, Number of cross-validation folds

    Returns:
    - results: pd.DataFrame, Accuracy comparison between original and constructed features
    """

    # Default models to test if not provided
    if models is None:
        models = {
            'LogisticRegression': LogisticRegression(),
            'DecisionTree': DecisionTreeClassifier(),
            'RandomForest': RandomForestClassifier()
        }

    # Preprocessing pipelines for original and constructed features
    def create_preprocessor(df):
        num_cols, cat_cols, other_cols = fz.extract_features(df, 'all')
        return ColumnTransformer(
            transformers=[
                ('num', Pipeline(steps=[
                    ('imputer', SimpleImputer(strategy='median')),
                    ('scaler', StandardScaler())
                ]), num_cols),
                ('cat', Pipeline(steps=[
                    ('imputer', SimpleImputer(strategy='most_frequent')),
                    ('onehot', OneHotEncoder(handle_unknown='ignore'))
                ]), cat_cols)
            ]
        )

    # Initialize LabelEncoder for target encoding
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)

    # Dictionary to store accuracy results
    accuracy_results = {
        'Model': [],
        'Average Accuracy with Original Features': [],
        'Average Accuracy with Constructed Features': []
    }

    # Iterate over each model and compare accuracies
    for model_name, model in models.items():
        # Accuracy with original features
        preprocessor_orig = create_preprocessor(X_original)
        X_trans_orig = preprocessor_orig.fit_transform(X_original)
        scores_orig = cross_val_score(model, X_trans_orig, y_encoded, cv=cv, scoring='accuracy')
        average_accuracy_orig = np.mean(scores_orig)

        # Accuracy with constructed features
        preprocessor_const = create_preprocessor(X_constructed)
        X_trans_const = preprocessor_const.fit_transform(X_constructed)
        scores_const = cross_val_score(model, X_trans_const, y_encoded, cv=cv, scoring='accuracy')
        average_accuracy_const = np.mean(scores_const)

        # Store the results
        accuracy_results['Model'].append(model_name)
        accuracy_results['Average Accuracy with Original Features'].append(average_accuracy_orig)
        accuracy_results['Average Accuracy with Constructed Features'].append(average_accuracy_const)

    # Convert the results into a DataFrame for easy viewing
    results_df = pd.DataFrame(accuracy_results)

    # Sort the results by accuracy with original features in descending order
    results_df = results_df.sort_values(by='Average Accuracy with Original Features', ascending=False)

    return results_df


In [ ]:
# Define the models to experiment with
models = {
    'Logistic Regression': LogisticRegression(),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Support Vector Machine': SVC(),
    'KNN': KNeighborsClassifier(),
    'Decision Trees': DecisionTreeClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42),
    'Extra Trees': ExtraTreesClassifier(random_state=42)
}

In [ ]:
X_original = df_cleaned.drop('loan_status', axis=1)  # Features before construction
X_constructed = df_constructed .drop('loan_status', axis=1) # Features after construction
y = df_cleaned['loan_status']

# Call the comparison function
results = compare_model_accuracy(X_original, X_constructed, y, models=models)

results

In [ ]:
# Splitting the data into features and target
X = df.drop('loan_status', axis=1)
y = df['loan_status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
# Label encode the target feature

# Initialize the LabelEncoder
le = LabelEncoder()

# Fit and transform the target feature
y_train = le.fit_transform(y_train)

In [ ]:
# Getting indices of numerical and categorical features of the X_train dataframe
indices_num_cols = X_train.columns.get_indexer(num_cols)
indices_cat_cols = X_train.columns.get_indexer(cat_cols)

In [ ]:
# Column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', StandardScaler(), indices_num_cols), # Scaling numerical columns
        ('encoder', OneHotEncoder(drop='first', sparse_output=False), indices_cat_cols) # OneHotEncoding categorical columns
    ],
    remainder='passthrough'
)

In [ ]:
# Pipeline
pipe = Pipeline([
    ('preprocessor', preprocessor)
])

In [ ]:
# Preprocessing and Transforming training data using pipeline
X_train_transformed = pipe.fit_transform(X_train)


### Handling imbalanced dataset


In [ ]:
# Oversampling the dataset using SMOTE
smote = SMOTE(random_state=42)  # Initialize SMOTE with optional random_state for reproducibility
X_train_transformed, y_train = smote.fit_resample(X_train_transformed, y_train)

In [ ]:
# Test
unique_values, counts = np.unique(y_train, return_counts=True)

# Print the counts of each class
for value, count in zip(unique_values, counts):
    print(f"Class {value}: {count}")


## Model selection (Before hyperparameter tuning)


In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression,
    'Random Forest': RandomForestClassifier,
    'Gradient Boosting': GradientBoostingClassifier,
    'Support Vector Machine': SVC,
    'KNN': KNeighborsClassifier,
    'Decision Trees': DecisionTreeClassifier,
    'Xgboost': XGBClassifier,
    'Extra Trees': ExtraTreesClassifier
}
# Define metric functions
metrics = {
    'accuracy': accuracy_score,
    'precision': precision_score,
    'recall': recall_score,
    'f1': f1_score
}

In [ ]:
def eval_models_across_metrics(models, metrics, X_train, y_train, cv=5, sort=False,
                               model_params=None):

    models_across_metrics = {metric: {} for metric in metrics}

    for metric in metrics:
        for model_name, model in models.items():

            if isinstance(model_params, dict):
                cv_scores = cross_val_score(model(**model_params[model_name]), X_train,
                                            y_train, cv=cv, scoring=metric)
            else:
                cv_scores = cross_val_score(model(), X_train,
                                            y_train, cv=cv, scoring=metric)

            cv_scores_mean = cv_scores.mean()

            models_across_metrics[metric][model_name] = round(cv_scores_mean, 3)

    if sort:
        for metric, model_scores in models_across_metrics.items():
            models_across_metrics[metric] = dict(
                sorted(model_scores.items(), key=lambda item: item[1], reverse=True)
                )

    return models_across_metrics

In [ ]:
# Comparing models across metrics
models_across_metrics = eval_models_across_metrics(models, metrics.keys(), X_train_transformed,
                                                   y_train, sort=True)
print(json.dumps(models_across_metrics, indent=4))

Conclusions:

After evaluating the metrics, I have decided to focus on the top 3 models: Random Forest Classifier, Extra Trees Classifier, Xgboost Classifier. These models have demonstrated strong performance across the different metrics, making them the best candidates for further fine-tuning and optimization.


## Hyperparameter tuning


In [ ]:
# Define top models for further hyperparameter tuning
models = {
    'Random Forest': RandomForestClassifier,
    'Xgboost': XGBClassifier,
    'Extra Trees': ExtraTreesClassifier
}

In [ ]:
# Define parameter grids
param_grids = {
    'Random Forest': {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    'Xgboost': {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 6, 10],
        'learning_rate': [0.01, 0.1, 0.2],
        'subsample': [0.8, 0.9, 1.0],
        'colsample_bytree': [0.8, 0.9, 1.0],
        'gamma': [0, 0.1, 0.2]
    },
    'Extra Trees': {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'bootstrap': [True, False]
    }
}

In [ ]:
def hyperparameter_tuning(models_dict = None, param_grids = None, X_train = None,
                          y_train = None, file_path=None, force_overwrite = False):
    def perform_grid_search(models_dict, param_grids, X_train, y_train):
        best_params = {}
        for model_name, model in models_dict.items():
            print(f"Processing {model_name}...")
            param_grid = param_grids[model_name]
            grid_search = GridSearchCV(estimator=model(), param_grid=param_grid,
                                    scoring='accuracy', cv=5, n_jobs=-1)
            grid_search.fit(X_train, y_train)
            best_params[model_name] = {
                'Best Parameters': grid_search.best_params_,
                'Average accuracy score on the best parameters': round(grid_search.best_score_, 3)
            }
        return best_params

    if file_path:
        if os.path.exists(file_path) and not force_overwrite:
            best_params = joblib.load(file_path)
        else:
            best_params = perform_grid_search(models_dict, param_grids, X_train, y_train)
            joblib.dump(best_params, file_path)
    else:
        best_params = perform_grid_search(models_dict, param_grids, X_train, y_train)

    return best_params

In [ ]:
# Finding best hyperparameters on top models using GridSearchCV
best_params = hyperparameter_tuning(file_path='best_params.joblib')
print(json.dumps(best_params, indent=4))

In [ ]:
# Comparing top models across metrics after hyperparameter tuning
models_across_metrics = eval_models_across_metrics(models, metrics.keys(), X_train_transformed,
                                                   y_train, sort=True, model_params=best_params)
print(json.dumps(models_across_metrics, indent=4))


## Model training


In [ ]:
estimators = []
for model_name, model in models.items():
    estimators.append((model_name, model(**best_params[model_name]['Best Parameters'])))

In [ ]:
def eval_voting_clf(estimators, X_train, y_train, cv = 5):
    # Create a voting classifier (hard voting)
    voting_clf_hard = VotingClassifier(estimators=estimators, voting='hard')

    # Create a voting classifier (soft voting)
    voting_clf_soft = VotingClassifier(estimators=estimators, voting='soft')

    # Apply cross-validation
    cv_scores_h = cross_val_score(voting_clf_hard, X_train, y_train, cv=cv, scoring='accuracy')
    cv_scores_s = cross_val_score(voting_clf_soft, X_train, y_train, cv=cv, scoring='accuracy')

    accuracy_results = {}

    accuracy_results['Hard Margin'] = round(cv_scores_h.mean(), 3)
    accuracy_results['Soft Margin'] = round(cv_scores_s.mean(), 3)

    return accuracy_results

In [ ]:
# Accuracy on hard and soft margin voting classifiers
accuracy = eval_voting_clf(estimators, X_train_transformed, y_train)
accuracy

In [ ]:
# Fit the best model
voting_clf = VotingClassifier(estimators, voting='hard')
voting_clf.fit(X_train_transformed, y_train)

## Saving

In [ ]:
# Save the Machine Learning model
joblib.dump(voting_clf, 'model.joblib')